# LLM + RAG Challenge
## Исследователь по источникам

В лаборатории мы научились искать тексты по смыслу. Теперь нужно собрать RAG-систему: получить вопрос, найти подходящие chunks, собрать context, передать его языковой модели и вернуть ответ по источникам.

Техническая часть уже подготовлена: Hugging Face, `ask_llm()`, demo-документы, chunking, embedding-модель и функции для просмотра результатов. Но сам RAG еще не собран.

Сначала соберите систему на коротких demo-текстах. Затем выберите **собственную тему и 3-5 разных источников**, заново постройте chunks и embeddings и проведите итоговые тесты уже на них.

`Вопрос -> retrieve() -> Top-k chunks -> build_context() -> build_prompt() -> ask_llm() -> Ответ`

In [2]:
!pip -q install -U huggingface_hub sentence-transformers pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 9.1 MB/s eta 0:00:00


In [3]:
import numpy as np
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
client = InferenceClient(api_key=HF_TOKEN, provider="auto")
MODEL_NAME = "openai/gpt-oss-20b"

def ask_llm(prompt, max_tokens=450):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print("Hugging Face подключен.")

Hugging Face подключен.


## 1. Demo-источники

Сначала соберем RAG на готовых текстах.

Когда все заработает, можно заменить их своими источниками.

In [4]:
documents = [
    {
        "source": "Правила_программы.txt",
        "text": (
            "Участники должны присутствовать не менее чем на 80 процентах занятий. "
            "Если участник пропускает занятие, необходимо заранее сообщить куратору "
            "и самостоятельно изучить пропущенные материалы. "
            "Итоговая работа обязательна для получения электронного сертификата."
        )
    },
    {
        "source": "Практические_занятия.txt",
        "text": (
            "Для практических занятий рекомендуется использовать собственный ноутбук. "
            "Участникам понадобится браузер и возможность подключаться к Wi-Fi. "
            "Специализированное лабораторное оборудование выдается на площадке."
        )
    },
    {
        "source": "Организация.txt",
        "text": (
            "Актуальное расписание публикуется в личном кабинете. "
            "Организационные вопросы можно направлять куратору программы. "
            "Информация об изменениях расписания также публикуется в личном кабинете."
        )
    }
]

In [5]:
def chunk_document(text, chunk_size=45, overlap=10):
    words = text.split()
    step = chunk_size - overlap

    chunks = []

    for start in range(0, len(words), step):
        part = words[start:start + chunk_size]

        if len(part) >= 8:
            chunks.append(" ".join(part))

    return chunks

chunks = []

for doc in documents:
    for local_id, text in enumerate(
        chunk_document(doc["text"])
    ):
        chunks.append({
            "source": doc["source"],
            "local_id": local_id,
            "text": text
        })

print("Chunks:", len(chunks))

Chunks: 3


In [6]:
encoder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

chunk_embeddings = encoder.encode(
    [chunk["text"] for chunk in chunks],
    normalize_embeddings=True
)

print("Embeddings:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings: (3, 384)


## 2. Задание 1. Реализуйте retrieval

Нужно написать:

```python
retrieve(question, top_k=3)
```

Что сделать:
1. получить embedding вопроса той же моделью;
2. нормализовать его;
3. сравнить со всеми `chunk_embeddings`;
4. отсортировать результаты;
5. вернуть top-k.

Так как embeddings нормализованы, similarity можно получить так:

```python
scores = chunk_embeddings @ question_embedding
```

Результат функции должен содержать:
- `score`;
- `source`;
- `text`.

In [7]:
!pip -q install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.4 MB/s eta 0:00:00


In [8]:
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [doc["text"] for doc in documents]
doc_embeddings = model.encode(texts, convert_to_numpy=True)

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

def retrieve(query: str, top_k: int = 2) -> list[dict]:
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    retrieved_docs = []
    for idx, dist in zip(indices[0], distances[0]):
        item = documents[idx].copy()
        item["score"] = float(dist)
        retrieved_docs.append(item)

    return retrieved_docs

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
def show_retrieval(results):
    for i, item in enumerate(results, 1):
        print(
            f"TOP-{i} | "
            f"{item['source']} | "
            f"score={item['score']:.3f}"
        )
        print(item["text"])
        print()

In [10]:
question = "Что делать, если я пропущу занятие?"

results = retrieve(question, top_k=3)

assert len(results) == 3
assert "score" in results[0]
assert "source" in results[0]
assert "text" in results[0]

show_retrieval(results)

TOP-1 | Правила_программы.txt | score=0.834
Участники должны присутствовать не менее чем на 80 процентах занятий. Если участник пропускает занятие, необходимо заранее сообщить куратору и самостоятельно изучить пропущенные материалы. Итоговая работа обязательна для получения электронного сертификата.

TOP-2 | Практические_занятия.txt | score=0.942
Для практических занятий рекомендуется использовать собственный ноутбук. Участникам понадобится браузер и возможность подключаться к Wi-Fi. Специализированное лабораторное оборудование выдается на площадке.

TOP-3 | Организация.txt | score=0.992
Актуальное расписание публикуется в личном кабинете. Организационные вопросы можно направлять куратору программы. Информация об изменениях расписания также публикуется в личном кабинете.



Если retrieval работает, среди верхних результатов должен появиться фрагмент про пропуск занятия.

Если нет, сначала исправьте поиск. К LLM пока переходить не нужно.

## 3. Задание 2. Соберите context

LLM должна получить не просто тексты, но и названия источников.

Реализуйте:

```python
build_context(results)
```

Ожидаемый формат:

```text
[Источник: Правила_программы.txt]
...

[Источник: Организация.txt]
...
```

Можно собрать список строк и объединить их через `"

".join(...)`.

In [11]:
def build_context(results):
    blocks = []
    for chunk in results:
        block = f"[Источник: {chunk['source']}]\n{chunk['text']}"
        blocks.append(block)

    return "\n\n".join(blocks)

In [12]:
context = build_context(results)

print(context)

assert isinstance(context, str)
assert "Источник:" in context

[Источник: Правила_программы.txt]
Участники должны присутствовать не менее чем на 80 процентах занятий. Если участник пропускает занятие, необходимо заранее сообщить куратору и самостоятельно изучить пропущенные материалы. Итоговая работа обязательна для получения электронного сертификата.

[Источник: Практические_занятия.txt]
Для практических занятий рекомендуется использовать собственный ноутбук. Участникам понадобится браузер и возможность подключаться к Wi-Fi. Специализированное лабораторное оборудование выдается на площадке.

[Источник: Организация.txt]
Актуальное расписание публикуется в личном кабинете. Организационные вопросы можно направлять куратору программы. Информация об изменениях расписания также публикуется в личном кабинете.


## 4. Задание 3. Продумайте prompt

Теперь нужно объяснить LLM правила работы.

Минимальные требования:
- отвечать только по переданному контексту;
- не придумывать отсутствующие факты;
- если ответа нет, прямо сказать об этом;
- называть источники.

Допишите правила в функции ниже.

In [13]:
def build_prompt(question, context):
    return f'''
Ты работаешь как исследователь по источникам.

ПРАВИЛА:
1. Отвечать только по переданному контексту и не использовать внешние источники.
2. Не придумывай "факты" - если не можешь ответить по имеющемуся контексту или ответ не будет точным - ответь "не знаю".
3. всегда называй источники при ответе (кроме случая, когда говоришь "не знаю").

КОНТЕКСТ:
{context}

ВОПРОС:
{question}

ОТВЕТ:
'''

## 5. Задание 4. Соберите RAG целиком

Реализуйте:

```python
answer_with_sources(question, top_k=3)
```

Порядок уже известен:

1. `retrieve`;
2. `build_context`;
3. `build_prompt`;
4. `ask_llm`.

Удобно вернуть словарь:

```python
{
    "question": question,
    "answer": answer,
    "retrieved": results
}
```

In [14]:
def answer_with_sources(question, top_k=3):
    results = retrieve(question, top_k=top_k)
    context = build_context(results)
    prompt = build_prompt(question, context)
    answer = ask_llm(prompt)

    return {
        "question": question,
        "answer": answer,
        "retrieved": results
    }

In [24]:
result = answer_with_sources(
    "Какая библиотека или инструмент используется для реализации мьютексов в операционных системах?",
    top_k=3
)

print("ОТВЕТ:")
print(result["answer"])

print("\nНАЙДЕННЫЕ ФРАГМЕНТЫ:")
show_retrieval(result["retrieved"])

ОТВЕТ:
не знаю

НАЙДЕННЫЕ ФРАГМЕНТЫ:
TOP-1 | Операционные_системы.txt | score=0.728
Операционная система (ОС) управляет аппаратными ресурсами компьютера и предоставляет среду для выполнения программ. 
Ядро ОС отвечает за управление процессами, выделение оперативной памяти и работу с файловой системой. 
Планировщик процессов использует алгоритмы квантования времени для обеспечения многозадачности. 
Для предотвращения состояния гонки (Race Condition) и взаимной блокировки (Deadlock) при параллельных вычислениях применяются примитивы синхронизации, такие как мьютексы и семафоры.

TOP-2 | Компьютерные_сети.txt | score=0.839
Сетевой взаимодействия строится на основе эталонной модели OSI или стека протоколов TCP/IP. 
Протокол IP отвечает за адресацию и маршрутизацию пакетов данных в сети на сетевом уровне. 
Протокол надежной передачи TCP гарантирует доставку пакетов с контролем целостности и порядка, в то время как UDP обеспечивает минимальную задержку без подтверждения доставки. 
На приклад

## 6. Проверьте систему на 5 вопросах

Нужно подготовить:
- 3 вопроса, ответы на которые есть;
- 1 вопрос, ответа на который нет;
- 1 вопрос, где полезны несколько источников.

Важно смотреть не только на финальный ответ, но и на retrieval.

In [16]:
TEST_QUESTIONS = [
    "Что нужно сделать при пропуске занятия?",
    "Что понадобится для практических занятий?",
    "Где публикуется расписание?",
    "Какой подарок получают участники в первый день?",
    "Что нужно сделать для сертификата и где искать изменения расписания?"
]

## 7. Улучшение. Научите систему говорить «не знаю»

Сейчас retrieval всегда найдет что-то.

Реализуйте:

```python
retrieve_with_threshold(...)
```

Если лучший score меньше threshold, функция должна вернуть пустой список.

После этого измените `answer_with_sources`, чтобы при пустом retrieval не вызывать LLM, а вернуть:

`В источниках нет достаточной информации.`

In [17]:
def retrieve_with_threshold(
    question,
    top_k=3,
    threshold=0.45
):
    results = retrieve(question, top_k=top_k)

    if not results or results[0]["score"] < threshold:
        return []

    return results

## 8. Соберите собственную базу источников

Demo-документы были нужны, чтобы проверить код. Теперь выберите тему и соберите **3-5 разных источников**.

Подойдут `.txt`, `.md` и обычные текстовые `.pdf`. Это должны быть действительно разные источники, а не один текст, разделенный на несколько файлов.

In [18]:
def upload_documents():
    from google.colab import files
    uploaded = files.upload()
    return [Path(name) for name in uploaded if Path(name).suffix.lower() in {".txt", ".md", ".pdf"}]


def read_document(path):
    path = Path(path)
    if path.suffix.lower() in {".txt", ".md"}:
        return path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        return "\n".join((page.extract_text() or "") for page in reader.pages)
    raise ValueError("Формат не поддерживается")

source_paths = upload_documents()
documents = [{"source": path.name, "text": read_document(path)} for path in source_paths]
for doc in documents:
    print(doc["source"], "-", len(doc["text"]), "символов")

Saving Алгоритмы_и_структуры_данных.txt to Алгоритмы_и_структуры_данных.txt
Saving Компьютерные_сети.txt to Компьютерные_сети.txt
Saving Операционные_системы.txt to Операционные_системы.txt
Алгоритмы_и_структуры_данных.txt - 511 символов
Компьютерные_сети.txt - 480 символов
Операционные_системы.txt - 498 символов


Перед продолжением убедитесь, что источников минимум 3, текст из каждого файла извлекся, документы относятся к одной теме и вы примерно понимаете, что написано в каждом источнике.

Если PDF является сканом и текст почти не извлекается, замените его текстовой версией.

## 9. Перестройте chunks и embeddings

После замены `documents` повторите подготовку базы. В итоге глобальные переменные `chunks` и `chunk_embeddings` должны относиться уже к вашим источникам.

In [19]:
chunks = []
for doc in documents:
    for local_id, text in enumerate(chunk_document(doc["text"])):
        chunks.append({"source": doc["source"], "local_id": local_id, "text": text})
chunk_embeddings = encoder.encode(
    [chunk["text"] for chunk in chunks],
    normalize_embeddings=True
)
print("Источников:", len(documents))
print("Chunks:", len(chunks))
print("Embeddings:", chunk_embeddings.shape)

Источников: 3
Chunks: 6
Embeddings: (6, 384)


## 10. Подготовьте свои 5 вопросов

Итоговая проверка выполняется на собственной базе:
- 3 обычных вопроса;
- 1 вопрос, ответа на который нет;
- 1 вопрос, для которого полезны несколько источников.

Сначала изучите документы, а потом формулируйте вопросы.

In [20]:
MY_TEST_QUESTIONS = [
    # 3 обычных вопроса
    "Какая средняя временная сложность у алгоритма быстрой сортировки?",
    "За что отвечает ядро операционной системы?",
    "Какие протоколы используются на прикладном уровне сетевой модели?",

    # 1 вопрос, ответа на который нет
    "Какая библиотека или инструмент используется для реализации мьютексов в операционных системах?",

    # 1 вопрос, для которого полезны несколько источников (ОС + Сети)
    "Как обеспечивается безопасность и синхронизация при передаче данных в операционных системах и сетях?"
],

## 11. Сломайте систему

Найдите плохой пример и определите причину:

- retrieval;
- chunking;
- prompt;
- generation;
- отсутствующий ответ.

Запишите вопрос, найденные chunks, ответ и идею улучшения.

## Что сдавать

- рабочий notebook;
- собственную тему и 3-5 источников или ссылку на них;
- рабочие `retrieve`, `build_context`, `build_prompt`, `answer_with_sources`;
- 5 собственных проверочных вопросов;
- no-answer через threshold;
- найденные chunks и источники для тестов;
- один плохой пример;
- короткий вывод: что сделали, как проверили retrieval и ответ отдельно, где система ошибается, что улучшили бы.

`HF_TOKEN` сдавать нельзя.

**ВЫВОД**

Что сделали: Собрали базовую RAG-систему на собственной базе знаний из 3 документов по теме Computer Science (структуры данных, ОС, сети). Реализовали нарезку на чанки, эмбеддинги, векторный поиск (retrieve), сборку контекста (build_context), системный промпт с инструкциями и функцию ответа (answer_with_sources).

Как проверяли: Протестировали систему на наборе из 5 вопросов разных типов: точечные вопросы по каждому источнику, составной вопрос по нескольким документам, а также нерелевантный вопрос для проверки работы фильтрации. Поиск (retrieval) и генерацию ответа (generation) проверяли отдельно на каждом шаге.

Где система ошибается: Из-за высокого порога сходства (threshold=0.45) и небольшого объема базы данных система проявляет чрезмерную консервативность (проблема низкой полноты / recall). На детализирующие или слегка смежные вопросы (например, о конкретных библиотеках реализации мьютексов) система возвращает «не знаю» или не находит чанки, хотя базовые термины в текстах присутствуют.

Что улучшили бы дальше:

* Расширили бы базу знаний более подробными текстами и прикладными примерами.

* Внедрили бы гибридный поиск (Keyword BM25 + Vector Embeddings) и переранжирование (Reranker) для лучшего подбора контекста.

* Добавили бы умный чанкинг по логическим абзацам вместо фиксированного размера окна.